## Read in cp3


In [1]:
import pandas as pd
import re
import os

In [2]:
INPUT_CSV = (
   "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint3_geocoded_2526.csv"
)

OUTPUT_CSV = (
    "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint4_cleaned_addresses.csv"
)
df = pd.read_csv(INPUT_CSV, low_memory=False)

print(f"Loaded: {INPUT_CSV}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

display(df.head())
with pd.option_context('display.max_rows', None):
    print(df['Location'].value_counts())




Loaded: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint3_geocoded_2526.csv
Rows: 26,502
Columns: 17


,Incident #,Date,Type,Location,Arrested,Name,DOB,Charges,Disposition,Location Prefix,raw_address,cleaned_address,latitude,longitude,geocode_confidence,geocoded_at,coord_valid
0,26025341,2026-06-01 23:53:00,NOISE ORDINANCE,ALLSTON ST & HOWARD ST,No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,ALLSTON ST & HOWARD ST,"ALLSTON ST & HOWARD ST, Lawrence, MA",42.711489,-71.146258,8.0,2026-06-28T02:47:17.094562+00:00,True
1,26025340,2026-06-01 23:52:00,SEX OFFENSE,75 MANCHESTER ST,No,NaN,NaN,NaN,REPORT REQUIRED,NaN,75 MANCHESTER ST,"75 MANCHESTER ST, Lawrence, MA",42.712262,-71.179138,10.0,2026-06-27T19:47:43.969354+00:00,True
2,26025338,2026-06-01 23:29:00,EXTRA CHECK,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST",No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST","SOUTH LAWRENCE EAST SCHOOL, Lawrence, MA",42.707040,-71.163110,7.0,2026-06-30T15:19:59.495398+00:00,True
3,26025337,2026-06-01 23:16:00,911 HANG-UP,98 OSGOOD ST,No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,98 OSGOOD ST,"98 OSGOOD ST, Lawrence, MA",42.696509,-71.148493,10.0,2026-06-28T02:37:44.357795+00:00,True
4,26025336,2026-06-01 23:15:00,MV STOP,20 ISLAND ST,No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,20 ISLAND ST,"20 ISLAND ST, Lawrence, MA",42.706226,-71.149554,10.0,2026-06-20T19:58:08.588338+00:00,True


Location
MCDONALD'S, 50 BROADWAY                                                 232
LPD, LAWRENCE POLICE, 90 LOWELL ST                                      184
WASHINGTON MILL, 240 CANAL ST                                           180
S UNION ST                                                              163
CVS, 266 BROADWAY                                                       136
BUCKLEY GARAGE, 99 AMESBURY ST                                          118
LPD, LAWRENCE POLICE, 55 LOWELL ST                                      104
TRAIN STATION - MCGOVERN TRANSPORTATION CENTER, 211 MERRIMACK ST         99
PACIFIC MILLS LOFTS, 300 CANAL ST                                        95
MARKET BASKET, 700 ESSEX ST                                              94
550 BROADWAY                                                             92
280 MERRIMACK ST                                                         90
SOUTH COMMON, MARKET & S UNION                                           90
PAC

## Clean up stuff we missed before the api calls

In [3]:
def clean_address_after_merge(address):
    """
    Clean the Location value after the geocoding merge.

    The function:
    - Converts text to uppercase
    - Removes periods, apostrophes, and backticks
    - Removes unwanted police-report prefixes/labels
    - Converts STREET to ST
    - Removes floor, apartment, and unit identifiers
    - Removes '#' and the characters attached to it until whitespace
    - Normalizes whitespace
    """
    if pd.isna(address):
        return ""

    address = str(address).upper().strip()

    # ---------------------------------------------------------
    # 1. Remove unwanted police-report terms
    # ---------------------------------------------------------
    unwanted_patterns = [
    r"\bREC\s*/\s*STOLEN\s+ARREST\b",
    r"\bARREST\s*/",
    r"\bV\s+TRUST\s*/",
    r"\bV\s+STOP\s*/",
    r"\bTOWED\b",
    r"\bWALK\s+IN\b",
    r"\bWALKIN\s*/",
    r"\bTOW\s*/",
    r"\bSTOP\s*/",
    r"\bV\s*/",
]

    for pattern in unwanted_patterns:
        address = re.sub(
            pattern,
            " ",
            address,
            flags=re.IGNORECASE,
        )
    
# Remove anything inside parentheses (including the parentheses)
# Examples:
# (REAR)
# (A)
# (PARKING LOT)
    address = re.sub(r"\([^)]*\)", " ", address)
    # Remove punctuation characters
# Examples:
# 150` ANDOVER ST
# 217 E HAVERHILL ST'
# 17 CYPRESS AVE.
# 23 BROADWAY°
    address = re.sub(r"[.'`°]", "", address)

    # ---------------------------------------------------------
    # 3. Standardize STREET to ST
    # ---------------------------------------------------------
    # 473 HAVERHILL STREET -> 473 HAVERHILL ST
    address = re.sub(
        r"\bSTREET\b",
        "ST",
        address,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 4. Remove floor identifiers
    # ---------------------------------------------------------
    # Handles:
    # 218 PARK ST FL 2
    # 300 CANAL ST FL3
    # 300 CANAL ST FLOOR 3
    #
    # Only removes the floor identifier, not everything after it.
    address = re.sub(
    r"\b(?:FLOOR\s*#?\s*[A-Z0-9-]+|FL\s*#?\s*\d+[A-Z0-9-]*)\b",
    " ",
    address,
    flags=re.IGNORECASE,
)

    # Remove FL or FLOOR when no value follows
   

    # ---------------------------------------------------------
    # 5. Remove apartment and unit identifiers
    # ---------------------------------------------------------
    # Handles:
    # APT 2
    # APT2
    # APARTMENT 4B
    # UNIT #5
    address = re.sub(
        r"\b(?:APT|APARTMENT|UNIT)\s*#?\s*[A-Z0-9-]+\b",
        " ",
        address,
        flags=re.IGNORECASE,
    )

    # Remove standalone APT, APARTMENT, or UNIT
    address = re.sub(
        r"\b(?:APT|APARTMENT|UNIT)\b",
        " ",
        address,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 6. Remove '#' and anything attached to it until whitespace
    # ---------------------------------------------------------
    # Examples:
    # 218 BROADWAY #1ST       -> 218 BROADWAY
    # 300 CANAL ST #311 FL3   -> 300 CANAL ST FL3
    #
    # FL3 is then removed by the floor rule above.
    address = re.sub(
        r"#\S*",
        " ",
        address,
    )

    # Remove any standalone number signs that remain
    address = address.replace("#", " ")

    # ---------------------------------------------------------
    # 7. Normalize whitespace
    # ---------------------------------------------------------
    address = re.sub(r"\s+", " ", address).strip()

    return address

## TEST

In [4]:
test_addresses = [
    "150` ANDOVER ST",
    "150` ARLINGTON ST",
    "17 CYPRESS AVE.",
    "217 E HAVERHILL ST'",
    "218 BROADWAY #1ST",
    "218 PARK ST FL 2",
    "300 CANAL ST #311 FL3",
    "473 HAVERHILL STREET",
    "V STOP / 250 BROADWAY",
    "STOP / 250 BROADWAY",
    "V / 250 BROADWAY",
    "V TRUST / 250 BROADWAY",
    "TOW/ 250 BROADWAY",
    "TOWED 250 BROADWAY",
    "WALK IN 250 BROADWAY",
    "REC/STOLEN ARREST 250 BROADWAY",
]

test_results = pd.DataFrame({
    "original": test_addresses,
    "cleaned": [
        clean_address_after_merge(address)
        for address in test_addresses
    ],
})

display(test_results)

,original,cleaned
0,150` ANDOVER ST,150 ANDOVER ST
1,150` ARLINGTON ST,150 ARLINGTON ST
2,17 CYPRESS AVE.,17 CYPRESS AVE
3,217 E HAVERHILL ST',217 E HAVERHILL ST
4,218 BROADWAY #1ST,218 BROADWAY
5,218 PARK ST FL 2,218 PARK ST
6,300 CANAL ST #311 FL3,300 CANAL ST
7,473 HAVERHILL STREET,473 HAVERHILL ST
8,V STOP / 250 BROADWAY,250 BROADWAY
9,STOP / 250 BROADWAY,250 BROADWAY


In [5]:
cleaned_df = df.copy()

cleaned_df["cleaned_location"] = (
    cleaned_df["Location"]
    .apply(clean_address_after_merge)
)

print(f"Rows before cleaning: {len(df):,}")
print(f"Rows after cleaning: {len(cleaned_df):,}")

Rows before cleaning: 26,502
Rows after cleaning: 26,502


## Compare and consolidate addresses 

In [6]:
comparison = cleaned_df[
    ["Location", "cleaned_location"]
].drop_duplicates()

changed = comparison[
    comparison["Location"].fillna("").astype(str).str.strip()
    != comparison["cleaned_location"]
]

print(f"Unique original locations: {cleaned_df['Location'].nunique(dropna=True):,}")
print(f"Unique cleaned locations: {cleaned_df['cleaned_location'].nunique(dropna=True):,}")
print(f"Unique values changed by cleaning: {len(changed):,}")

display(changed.head(50))

Unique original locations: 8,615
Unique cleaned locations: 8,428
Unique values changed by cleaning: 320


,Location,cleaned_location
42,"MCDONALD'S, 599 ANDOVER ST","MCDONALDS, 599 ANDOVER ST"
95,"RIVERWALK APARTMENTS, 1 S UNION ST","RIVERWALK , 1 S UNION ST"
108,"MANCHESTER STREET PARK, 77 MANCHESTER ST","MANCHESTER ST PARK, 77 MANCHESTER ST"
121,"PEMBERTON PARK, 400 CANAL STREET","PEMBERTON PARK, 400 CANAL ST"
149,"MCDONALD'S, 50 BROADWAY","MCDONALDS, 50 BROADWAY"
154,"S.E.S SCHOOL, 233 HAVERHILL ST","SES SCHOOL, 233 HAVERHILL ST"
172,"HAFFNER'S GAS STATION, 69 PARKER ST","HAFFNERS GAS STATION, 69 PARKER ST"
214,"PAISANO'S PIZZERIA, 265 MERRIMACK ST","PAISANOS PIZZERIA, 265 MERRIMACK ST"
225,"NICK'S PLACE, 195 LOWELL ST","NICKS PLACE, 195 LOWELL ST"
228,"LAWRENCE SENIOR CENTER, 155 HAVERHILL STREET","LAWRENCE SENIOR CENTER, 155 HAVERHILL ST"


In [7]:
collapsed = (
    cleaned_df.loc[
        cleaned_df["cleaned_location"].ne(""),
        ["cleaned_location", "Location"],
    ]
    .drop_duplicates()
    .groupby("cleaned_location")["Location"]
    .agg(list)
    .reset_index()
)

collapsed = collapsed[
    collapsed["Location"].apply(len) > 1
].copy()

collapsed["original_address_count"] = (
    collapsed["Location"].apply(len)
)

collapsed = collapsed.sort_values(
    "original_address_count",
    ascending=False,
)

print(f"Collapsed address groups: {len(collapsed):,}")

display(collapsed.head(50))

Collapsed address groups: 185


,cleaned_location,Location,original_address_count
7138,ESSEX ST,"[ESSEX ST, ESSEX STREET, ESSEX ST.]",3
3244,300 CANAL ST,"[300 CANAL ST, 300 CANAL ST., 300 CANAL STREET]",3
28,1 DOYLE ST,"[1 DOYLE ST, 1 DOYLE STREET]",2
6971,COMMON ST & BROADWAY,"[COMMON ST & BROADWAY, COMMON STREET & BROADWAY]",2
6728,"BELLESINI ACADEMY, 94 BRADFORD ST","[BELLESINI ACADEMY, 94 BRADFORD ST, BELLESINI ...",2
6810,"BRIGHT BRIDGE CREDIT UNION, 500 MERRIMACK ST","[BRIGHT BRIDGE CREDIT UNION, 500 MERRIMACK ST,...",2
6820,BROADWAY & CROSS ST,"[BROADWAY & CROSS ST, BROADWAY & CROSS STREET]",2
6835,BROADWAY & WATER ST,"[BROADWAY & WATER ST, BROADWAY & WATER STREET]",2
6865,"BUCKLEY GARAGE, 99 AMESBURY ST","[BUCKLEY GARAGE, 99 AMESBURY ST, BUCKLEY GARAG...",2
6925,"CENTRAL CATHOLIC HIGH SCHOOL, 300 HAMPSHIRE ST","[CENTRAL CATHOLIC HIGH SCHOOL, 300 HAMPSHIRE S...",2


In [8]:
with pd.option_context('display.max_rows', None):
    print(cleaned_df['Location'].value_counts())
coordinate_columns = [
    column
    for column in ["latitude", "longitude", "lat", "long"]
    if column in cleaned_df.columns
]

print("Coordinate columns found:", coordinate_columns)

if coordinate_columns:
    print(
        cleaned_df[coordinate_columns]
        .notna()
        .sum()
    )
display(cleaned_df.head())

Location
MCDONALD'S, 50 BROADWAY                                                 232
LPD, LAWRENCE POLICE, 90 LOWELL ST                                      184
WASHINGTON MILL, 240 CANAL ST                                           180
S UNION ST                                                              163
CVS, 266 BROADWAY                                                       136
BUCKLEY GARAGE, 99 AMESBURY ST                                          118
LPD, LAWRENCE POLICE, 55 LOWELL ST                                      104
TRAIN STATION - MCGOVERN TRANSPORTATION CENTER, 211 MERRIMACK ST         99
PACIFIC MILLS LOFTS, 300 CANAL ST                                        95
MARKET BASKET, 700 ESSEX ST                                              94
550 BROADWAY                                                             92
280 MERRIMACK ST                                                         90
SOUTH COMMON, MARKET & S UNION                                           90
PAC

,Incident #,Date,Type,Location,Arrested,Name,DOB,Charges,Disposition,Location Prefix,raw_address,cleaned_address,latitude,longitude,geocode_confidence,geocoded_at,coord_valid,cleaned_location
0,26025341,2026-06-01 23:53:00,NOISE ORDINANCE,ALLSTON ST & HOWARD ST,No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,ALLSTON ST & HOWARD ST,"ALLSTON ST & HOWARD ST, Lawrence, MA",42.711489,-71.146258,8.0,2026-06-28T02:47:17.094562+00:00,True,ALLSTON ST & HOWARD ST
1,26025340,2026-06-01 23:52:00,SEX OFFENSE,75 MANCHESTER ST,No,NaN,NaN,NaN,REPORT REQUIRED,NaN,75 MANCHESTER ST,"75 MANCHESTER ST, Lawrence, MA",42.712262,-71.179138,10.0,2026-06-27T19:47:43.969354+00:00,True,75 MANCHESTER ST
2,26025338,2026-06-01 23:29:00,EXTRA CHECK,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST",No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST","SOUTH LAWRENCE EAST SCHOOL, Lawrence, MA",42.707040,-71.163110,7.0,2026-06-30T15:19:59.495398+00:00,True,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST"
3,26025337,2026-06-01 23:16:00,911 HANG-UP,98 OSGOOD ST,No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,98 OSGOOD ST,"98 OSGOOD ST, Lawrence, MA",42.696509,-71.148493,10.0,2026-06-28T02:37:44.357795+00:00,True,98 OSGOOD ST
4,26025336,2026-06-01 23:15:00,MV STOP,20 ISLAND ST,No,NaN,NaN,NaN,SERVICED - SEE NOTES,NaN,20 ISLAND ST,"20 ISLAND ST, Lawrence, MA",42.706226,-71.149554,10.0,2026-06-20T19:58:08.588338+00:00,True,20 ISLAND ST


In [9]:
cleaned_df = cleaned_df.drop(columns=["Source PDF"], errors="ignore")

In [10]:
# Replace Location with the cleaned version
cleaned_df["Location"] = cleaned_df["cleaned_location"]

# Remove the temporary column
cleaned_df = cleaned_df.drop(columns=["cleaned_location"])

## Create cp4

In [11]:
os.makedirs(
    os.path.dirname(OUTPUT_CSV),
    exist_ok=True,
)

cleaned_df.to_csv(
    OUTPUT_CSV,
    index=False,
)

print(f"Saved: {OUTPUT_CSV}")
print(f"Saved rows: {len(cleaned_df):,}")

Saved: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/checkpoint4_cleaned_addresses.csv
Saved rows: 26,502


## Verify 

In [12]:
verification_df = pd.read_csv(
    OUTPUT_CSV,
    low_memory=False,
)

print(f"Reloaded rows: {len(verification_df):,}")
print(f"Reloaded columns: {len(verification_df.columns):,}")
print(
    "cleaned_location present:",
    "cleaned_location" in verification_df.columns,
)

display(
    verification_df[
        ["Location"]
    ].head(20)
)

Reloaded rows: 26,502
Reloaded columns: 17
cleaned_location present: False


,Location
0,ALLSTON ST & HOWARD ST
1,75 MANCHESTER ST
2,"SOUTH LAWRENCE EAST SCHOOL, 165 CRAWFORD ST"
3,98 OSGOOD ST
4,20 ISLAND ST
5,77 OXFORD ST
6,101 BENNINGTON ST
7,HAVERHILL ST & BROADWAY
8,237 ERVING AVE
9,ESSEX ST & UNION ST
